# CSE 151B Competition Notebook

Works on **Google Colab (T4)**, **Kaggle (T4/P100)**, and **UCSD DataHub** with a GPU runtime.

Pipeline:
1. Install dependencies
2. Load dataset
3. Run inference
4. Save submission CSV

The single entry point required by the competition is `run_inference()` in Section 5.

## 1. Environment Setup

**Run this cell once per session.**
After installation, **restart the runtime** (Runtime -> Restart runtime on Colab, or Kernel -> Restart on Kaggle/DataHub) before continuing.

> On Colab: Runtime -> Change runtime type -> T4 GPU (before running anything)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.51.3", "protobuf>=4.0.0", "bitsandbytes", "accelerate",
    "sympy", "numpy", "tqdm", "huggingface_hub"], check=True)
print("Done - restart runtime now")

## 2. Verify GPU

Run this after restarting the runtime. Confirms CUDA is visible and vLLM imports correctly.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. On Colab: Runtime -> Change runtime type -> T4 GPU. "
    "On Kaggle: Settings -> Accelerator -> GPU."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")


## 3. HuggingFace Login

Qwen3-4B is a gated model - you need a HuggingFace account and must accept the model license at  
https://huggingface.co/Qwen/Qwen3-4B

**On Colab**: add your token as a Secret named `HF_TOKEN` (key icon in the left sidebar), then run the cell below.  
**On Kaggle**: add it under Add-ons -> Secrets.  
**On DataHub**: paste it directly (don't commit to git).

In [ ]:
import os

HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("Token loaded from Colab secrets.")
except Exception:
    pass

if HF_TOKEN is None:
    # Kaggle secret
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("Token loaded from environment variable.")

if HF_TOKEN is None:
    # Fallback: paste directly (do NOT commit this to git)
    HF_TOKEN = ""  # <- paste your token here if not using secrets

assert HF_TOKEN, "HF_TOKEN is empty. Add it as a Colab/Kaggle secret or paste it above."

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("HuggingFace login successful.")

## 4. Configuration

All hyperparameters are set here - these are the **final values used for submission**.

In [ ]:
import json, csv, re, os
from pathlib import Path
from typing import Optional
from tqdm import tqdm

# -- Paths
DATA_PATH   = "data/private.jsonl"   # private test set (no answers)
OUTPUT_PATH = "results/submission.csv"

# -- Model
MODEL_NAME  = "Qwen/Qwen3-4B"        # HuggingFace model ID

# -- Inference hyperparameters
MAX_TOKENS  = 32768
TEMPERATURE = 0.2
TOP_P       = 0.85
TOP_K       = 20

print(f"Model       : {MODEL_NAME}")
print(f"Data        : {DATA_PATH}")
print(f"Output      : {OUTPUT_PATH}")
print(f"Max tokens  : {MAX_TOKENS}")
print(f"Temperature : {TEMPERATURE}")

## 5. Prompts & Answer Extraction

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step.\n\n"

    " FINAL ANSWER FORMAT\n"
    "At the very LAST LINE of your response, place ALL answers inside exactly ONE \\boxed{}.\n"
    "  DO:     \\boxed{380, 315, 13, 310}  (all parts, comma-separated, one box)\n"
    "  DO:     \\boxed{5/8}  (single answer)\n"
    "  DON'T:  box each sub-answer in a separate \\boxed{} throughout the solution\n"
    "  DON'T:  \\boxed{380}  ...text...  \\boxed{315}  ...text...  \\boxed{13}\n"
    "Even if you use \\boxed{} for intermediate steps during working, you MUST finish with "
    "a single combined \\boxed{a, b, c} on the very last line - all answers, in the order asked.\n"
    "Never leave \\boxed{} empty.\n\n"

    "EXACT FORM RULES\n"
    "1. SYMBOLIC OVER NUMERIC - if the answer is a function applied to given constants, "
    "write the expression, NOT a decimal:\n"
    "  DO:     \\arctan(4.76)          DON'T: 1.3635\n"
    "  DO:     \\ln(0.5)/\\ln(0.96584)  DON'T: 19.94\n"
    "  DO:     (1/2)^{(1999-1963)/31} DON'T: 0.447\n"
    "2. DECIMAL PRECISION: when a decimal is required, give at minimum 6 significant digits:\n"
    "  DO:     7.79744   DON'T: 7.80  |  DO: 442.857   DON'T: 442.86  |  DO: 12.0814  DON'T: 12.08\n"
    "3. PRESERVE STRUCTURE: if the problem writes 2*8*x, write 2*8*x not 16x.\n"
    "4. EXPLICIT MULTIPLICATION  write 3*t*(1-t)^2, not 3t(1-t)^2.\n"
    "5. EXPONENTIALS: write \\exp(0.016*t) or e^{0.016t}, not standalone e^0.016t.\n"
    "6. FRACTIONS: use exact fractions (5/8) for rational results.\n"
    "7. ORDER: answer multi-part questions in the exact order the problem asks.\n"
    "8. NO ANGLE BRACKETS: do not wrap answers in <> brackets.\n\n"

    "Before writing the final \\boxed{}, verify your answer satisfies the original problem. "
    "Commit to your best answer."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices carefully, then select the single best answer.\n\n"
    "STEP 1 Solve: Work through the problem step-by-step to derive your answer.\n"
    "STEP 2 Match: Compare your result against every option:\n"
    "  a) Check algebraic/symbolic equivalence (e.g. pi*sqrt(a) = pi*a^{1/2}, "
    "4/3*ln(3) = 2/3*ln(9), 1-cos^2(x) = sin^2(x)).\n"
    "  b) If options look different, plug in a concrete numeric value for any free variable "
    "and evaluate BOTH your answer and each option and pick the one whose value matches yours.\n"
    "  c) If two options appear numerically equal, prefer the one whose algebraic form "
    "matches your derivation most directly.\n"
    "STEP 3 Commit: Trust your derivation. Do not abandon a correct answer just because "
    "the option looks different in form.\n\n"
    "You MUST always pick one of the given letters, never say none match. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_messages(question: str, options: Optional[list]) -> list[dict]:
    """Return a messages list in OpenAI chat format for vLLM."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        user_content = f"{question}\n\nOptions:\n{opts_text}"
        system = SYSTEM_PROMPT_MCQ
    else:
        user_content = question
        system = SYSTEM_PROMPT_MATH
    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": user_content},
    ]


def extract_boxed(text: str) -> str:
    """Extract content of the LAST \\boxed{} in text."""
    # Handles nested braces correctly
    matches = []
    i = 0
    while i < len(text):
        idx = text.find(r'\boxed{', i)
        if idx == -1:
            break
        # Find matching closing brace
        depth = 0
        j = idx + len(r'\boxed{')
        start = j
        while j < len(text):
            if text[j] == '{':
                depth += 1
            elif text[j] == '}':
                if depth == 0:
                    matches.append(text[start:j])
                    break
                depth -= 1
            j += 1
        i = idx + 1
    return matches[-1].strip() if matches else ""


print("Prompts and extraction utilities loaded.")

## 6. `run_inference()` - Competition Entry Point

This is the single function required by the competition spec.  
Call `run_inference()` to reproduce the full pipeline end-to-end.

In [ ]:
from transformers import BitsAndBytesConfig
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def _write_csv(out_path: Path, data: list, done: dict) -> None:
    """Write id,response CSV in original dataset order."""
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "response"])
        writer.writeheader()
        for item in data:
            writer.writerow({
                "id":       item["id"],
                "response": done.get(item["id"], ""),
            })


def run_inference(
    data_path:   str = DATA_PATH,
    output_path: str = OUTPUT_PATH,
    model_name:  str = MODEL_NAME,
) -> str:

    # 1. Load dataset
    print("Loading dataset...")
    data = [json.loads(line) for line in open(data_path, encoding="utf-8")]
    n_mcq  = sum(bool(d.get("options")) for d in data)
    n_free = len(data) - n_mcq
    print(f"Loaded {len(data)} questions ({n_mcq} MCQ, {n_free} free-form)")

    # 2. Resume
    out_path = Path(output_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    done: dict[int, str] = {}
    if out_path.exists():
        with open(out_path, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                resp = row["response"]
                if resp and not resp.startswith("ERROR:"):
                    done[int(row["id"])] = resp
        print(f"Resuming: {len(done)} done, {len(data) - len(done)} remaining")

    remaining = [d for d in data if d["id"] not in done]
    if not remaining:
        print("All questions already completed. Writing CSV.")
        _write_csv(out_path, data, done)
        return str(out_path)

    # 3. Load tokenizer & model
    print(f"\nLoading tokenizer: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    print(f"Loading model: {model_name}  (this takes ~2-4 min)")

    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    )
    model.eval()
    print("Model loaded.")

    # 4. Inference loop
    print(f"\nRunning inference on {len(remaining)} questions...")
    for item in tqdm(remaining, desc="Generating"):
        messages = build_messages(item["question"], item.get("options"))
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        try:
            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=MAX_TOKENS,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    top_k=TOP_K,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
            # Decode only the newly generated tokens
            new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
            response = tokenizer.decode(new_tokens, skip_special_tokens=True)
        except Exception as e:
            print(f"\nERROR on id={item['id']}: {e}")
            response = f"ERROR: {e}"

        done[item["id"]] = response

        # Save after every question so progress isn't lost on disconnect
        _write_csv(out_path, data, done)

    n_empty = sum(1 for d in data if not done.get(d["id"], "").strip())
    print(f"\nDone. {len(data)} questions processed, {n_empty} empty responses.")
    print(f"Submission saved to: {out_path}")
    return str(out_path)

## 7. Run

**Full run** (all questions):
```python
run_inference()
```

**Quick test** on a subset (to verify the pipeline before committing to the full run):
```python
run_inference(data_path="data/private.jsonl")  # limit handled below
```

Set `TEST_MODE = True` to run on 20 questions only.

In [ ]:
TEST_MODE = False

if TEST_MODE:
    # Write a 20-question subset to a temp file and run on that
    import tempfile
    data_all = [json.loads(line) for line in open(DATA_PATH, encoding="utf-8")]
    subset   = data_all[:20]
    tmp      = tempfile.NamedTemporaryFile(mode="w", suffix=".jsonl",
                                           delete=False, encoding="utf-8")
    for item in subset:
        tmp.write(json.dumps(item) + "\n")
    tmp.close()
    print(f"TEST MODE: running on 20 questions -> {tmp.name}")
    run_inference(data_path=tmp.name, output_path="results/test_run.csv")
else:
    run_inference()